# CS4241 - Introduction to Artificial Intelligence
## Part D: Full RAG Pipeline Implementation (10 Marks)

**Name:** Maureen Amago  
**Index Number:** 10022200180

---
### Pipeline Architecture
```
User Query
    ↓
[STAGE 1] Retrieval       — Search vector store, get top-k chunks
    ↓
[STAGE 2] Context Selection — Rank & filter to fit token budget
    ↓
[STAGE 3] Prompt Builder   — Inject context + strict rules
    ↓
[STAGE 4] LLM              — Ollama (llama3) or smart simulation
    ↓
[STAGE 5] Response         — Display answer + pipeline metrics
```
Each stage is **logged** with timestamps so every decision is traceable.

---
## Cell 1: Imports & Logger

In [13]:
# Name: Maureen Amago | Index: 10022200180
import re, math, time, logging
import numpy as np
import pandas as pd
from datetime import datetime
from pypdf import PdfReader
from collections import Counter

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('RAG')
print('✅ Logger ready.')

✅ Logger ready.


---
## Cell 2: Load Document & Create Chunks

In [14]:
# Name: Maureen Amago | Index: 10022200180
log.info('Loading 2025 Ghana Budget PDF...')
t0 = time.time()

reader = PdfReader('2025-Budget-Statement-and-Economic-Policy_v4.pdf')
raw_text = ''
for i in range(min(50, len(reader.pages))):
    p = reader.pages[i].extract_text()
    if p: raw_text += p + ' '

clean_text = re.sub(r'\s+', ' ', raw_text)
clean_text = re.sub(r'[^\x00-\x7F]+', ' ', clean_text).strip()

# Remove table-of-contents noise lines (lines that are mostly dots)
clean_text = re.sub(r'[.]{4,}', ' ', clean_text)

# Chunk: 500 chars, 50 char overlap
chunks = [clean_text[i:i+500] for i in range(0, len(clean_text), 450)]

log.info(f'Loaded {len(clean_text):,} chars → {len(chunks)} chunks in {time.time()-t0:.2f}s')
print(f'Total chunks : {len(chunks)}')
print(f'First chunk  : {chunks[0][:200]}...')

13:33:01 | INFO | Loading 2025 Ghana Budget PDF...
13:33:04 | INFO | Loaded 98,990 chars → 220 chunks in 2.30s


Total chunks : 220
First chunk  : THEME: Resetting The Economy For The Ghana We Want The Budget Statement and Economic Policy of the Government of Ghana for the 2025 Financial Year Presented to Parliament by DR. CASSIEL ATO FORSON (MP...


---
## Cell 3: TF-IDF Vector Store

In [15]:
# Name: Maureen Amago | Index: 10022200180
log.info('Building TF-IDF vector store...')
t1 = time.time()

STOP = {'the','and','of','in','to','for','is','a','an','on','at','by','as','be','are','this','that','was','with'}

def tokenize(text):
    return [w for w in re.findall(r'\b\w{2,}\b', text.lower()) if w not in STOP]

class VectorStore:
    """Custom TF-IDF vector store with cosine similarity search."""
    def __init__(self, docs):
        self.docs = docs
        all_tok = tokenize(' '.join(docs))
        self.vocab = {w:i for i,(w,_) in enumerate(Counter(all_tok).most_common(5000))}
        self.idf = {w: math.log(len(docs)/(1+sum(1 for d in docs if w in d.lower())))+1
                    for w in self.vocab}
        self.vecs = np.array([self._embed(d) for d in docs])
        log.info(f'Vector store ready | vocab={len(self.vocab):,} | shape={self.vecs.shape}')

    def _embed(self, text):
        v = np.zeros(len(self.vocab))
        toks = tokenize(text)
        if not toks: return v
        for w,c in Counter(toks).items():
            if w in self.vocab:
                v[self.vocab[w]] = (c/len(toks)) * self.idf[w]
        return v

    def search(self, query, k=5):
        qv = self._embed(query)
        sims = []
        for i, cv in enumerate(self.vecs):
            d = np.linalg.norm(qv) * np.linalg.norm(cv)
            sims.append(float(np.dot(qv,cv)/d) if d>0 else 0.0)
        top = np.argsort(sims)[-k:][::-1]
        return [{'doc_id':int(i),'text':self.docs[i],'score':round(sims[i],4)} for i in top]

vs = VectorStore(chunks)
log.info(f'Vector store built in {time.time()-t1:.2f}s')

13:33:04 | INFO | Building TF-IDF vector store...
13:33:04 | INFO | Vector store ready | vocab=3,064 | shape=(220, 3064)
13:33:04 | INFO | Vector store built in 0.46s


---
## Cell 4: Full RAG Pipeline

In [16]:
# Name: Maureen Amago | Index: 10022200180
import requests

PROMPT_TEMPLATE = """You are a Ghana Budget AI specialist for 2025.
Use ONLY the context below to answer the question.
If the answer is NOT in the context, say: "I cannot find that information in the 2025 Budget document."

=== CONTEXT FROM BUDGET DOCUMENT ===
{context}
=====================================

Question: {query}

Answer (2-3 concise sentences based only on the context above):"""


def smart_simulate(context, query):
    """
    When Ollama is not available, produce a grounded answer from the context.
    Finds the most informative sentence in the context that relates to the query.
    """
    query_words = set(tokenize(query))
    # Split context into sentences
    sentences = re.split(r'(?<=[.!?]) +', context)
    # Score each sentence by how many query words it contains
    scored = []
    for s in sentences:
        s_clean = s.strip()
        if len(s_clean) < 30: continue   # skip short fragments
        overlap = len(query_words & set(tokenize(s_clean)))
        scored.append((overlap, s_clean))
    scored.sort(reverse=True)
    
    if not scored or scored[0][0] == 0:
        return "I cannot find that information in the 2025 Budget document."
    
    # Return the top 2 most relevant sentences
    best = [s for _, s in scored[:2]]
    return "Based on the 2025 Budget document: " + " ".join(best)


def call_llm(prompt, context, query):
    """Try Ollama (llama3). Fall back to smart simulation."""
    try:
        r = requests.post(
            'http://localhost:11434/api/generate',
            json={'model':'llama3','prompt':prompt,'stream':False},
            timeout=12
        )
        if r.status_code == 200:
            return r.json().get('response','').strip(), 'ollama:llama3'
    except: pass
    return smart_simulate(context, query), 'smart-simulation'


def rag_pipeline(user_query, k=5, top_k_context=2):
    pipeline_start = time.time()
    SEP = '='*65
    print(SEP)
    print(f'  RAG PIPELINE  |  {datetime.now().strftime("%H:%M:%S")}')
    print(SEP)
    print(f'  QUERY: {user_query}')
    print(SEP)

    # ── STAGE 1: RETRIEVAL ──────────────────────────────────
    log.info('STAGE 1 ▶ Retrieval started')
    t = time.time()
    results = vs.search(user_query, k=k)
    log.info(f'STAGE 1 ✓ Retrieved {k} docs in {time.time()-t:.3f}s')

    print(f'\n📂 STAGE 1 — RETRIEVED DOCUMENTS (Top {k})')
    print('-'*65)
    display(pd.DataFrame([{
        'Rank': i+1,
        'Doc ID': r['doc_id'],
        'Similarity Score': r['score'],
        'Preview': r['text'][:90].replace('\n',' ') + '...'
    } for i,r in enumerate(results)]))

    # ── STAGE 2: CONTEXT SELECTION ───────────────────────────
    log.info(f'STAGE 2 ▶ Selecting top {top_k_context} of {k} chunks')
    selected = results[:top_k_context]
    context  = '\n---\n'.join([r['text'] for r in selected])
    log.info(f'STAGE 2 ✓ Context length: {len(context)} chars')

    print(f'\n✂️  STAGE 2 — CONTEXT SELECTION (keeping top {top_k_context})')
    print('-'*65)
    for r in selected:
        print(f'  [Doc {r["doc_id"]}] Score={r["score"]}\n  {r["text"]}\n')

    # ── STAGE 3: PROMPT BUILDING ─────────────────────────────
    log.info('STAGE 3 ▶ Building prompt')
    prompt = PROMPT_TEMPLATE.format(context=context, query=user_query)
    log.info(f'STAGE 3 ✓ Prompt length: {len(prompt)} chars')

    print(f'\n📝 STAGE 3 — FINAL PROMPT SENT TO LLM')
    print('-'*65)
    print(prompt)

    # ── STAGE 4: LLM ─────────────────────────────────────────
    log.info('STAGE 4 ▶ Calling LLM...')
    t = time.time()
    answer, model = call_llm(prompt, context, user_query)
    log.info(f'STAGE 4 ✓ Response from [{model}] in {time.time()-t:.2f}s')

    # ── STAGE 5: RESPONSE ────────────────────────────────────
    total = time.time() - pipeline_start
    print(f'\n💬 STAGE 5 — FINAL RESPONSE')
    print('-'*65)
    print(answer)
    print(f'\n⏱️  Done in {total:.2f}s  |  Model: {model}')
    print(SEP + '\n')
    return answer

print('✅ RAG Pipeline ready!')

✅ RAG Pipeline ready!


---
## Cell 5: Interactive — Ask Your Own Question

In [17]:
# Name: Maureen Amago | Index: 10022200180
user_question = input('Ask the Ghana Budget AI: ')
rag_pipeline(user_question)

13:33:06 | INFO | STAGE 1 ▶ Retrieval started
13:33:06 | INFO | STAGE 1 ✓ Retrieved 5 docs in 0.004s


  RAG PIPELINE  |  13:33:06
  QUERY: 

📂 STAGE 1 — RETRIEVED DOCUMENTS (Top 5)
-----------------------------------------------------------------


,Rank,Doc ID,Similarity Score,Preview
0,1,219,0.0,ined by the high proportion of T -Bills issued...
1,2,68,0.0,(Table 2). Prior to falling to their current ...
2,3,79,0.0,acroeconomic reforms aimed at fiscal consolida...
3,4,78,0.0,(Not Achieved) 2 Inflation (annual average) ...
4,5,77,0.0,an improvement compared to the preceding year....


13:33:06 | INFO | STAGE 2 ▶ Selecting top 2 of 5 chunks
13:33:06 | INFO | STAGE 2 ✓ Context length: 945 chars
13:33:06 | INFO | STAGE 3 ▶ Building prompt
13:33:06 | INFO | STAGE 3 ✓ Prompt length: 1300 chars
13:33:06 | INFO | STAGE 4 ▶ Calling LLM...
13:33:06 | INFO | STAGE 4 ✓ Response from [smart-simulation] in 0.02s



✂️  STAGE 2 — CONTEXT SELECTION (keeping top 2)
-----------------------------------------------------------------
  [Doc 219] Score=0.0
  ined by the high proportion of T -Bills issued during the year. This resulted in higher domestic short -term financing and posed a high refinancing risk on the debt portfolio. 189. Mr. Speaker, following the restructuring of the Eurobonds, the weighted average interest rate reduced to 3.4 percent in 2024 from 5.3 percent in 2023. Also, about 85.4 percent of the debt portfolio is at a fixed interest rate compared to 92.1 percent in 2023.

  [Doc 68] Score=0.0
   (Table 2). Prior to falling to their current levels, oil prices peaked in mid -2022 at US$120 per barrel, according to historical data. 52. In addition, gold prices averaged US$ 2,388 per troy ounce in 2024, buoyed by safe - haven demand amidst geopolitical tensions and inflationary pressures. Projections for 2025 suggest prices will remain stable at US$2,325 per troy ounce, supported by sust

'I cannot find that information in the 2025 Budget document.'

---
## Cell 6: Pre-set Test Queries (for Examiner)

In [18]:
# Name: Maureen Amago | Index: 10022200180
test_queries = [
    'What is the projected economic growth for Ghana in 2025?',
    'What are the key revenue measures in the 2025 budget?',
    'What is the budget for the Ghana space program?'   # Hallucination test
]
for q in test_queries:
    rag_pipeline(q)

13:33:06 | INFO | STAGE 1 ▶ Retrieval started
13:33:06 | INFO | STAGE 1 ✓ Retrieved 5 docs in 0.003s


  RAG PIPELINE  |  13:33:06
  QUERY: What is the projected economic growth for Ghana in 2025?

📂 STAGE 1 — RETRIEVED DOCUMENTS (Top 5)
-----------------------------------------------------------------


,Rank,Doc ID,Similarity Score,Preview
0,1,59,0.3465,ese economies continue to struggle with struct...
1,2,70,0.3225,"1,800 1,801 1,850 2,388 2,325 3 Cocoa Beans (M..."
2,3,60,0.2887,2025 Budget 6 Table 1: Global Economic Growth ...
3,4,16,0.2233,Receipts in 2024 (US$Mn) 41 Table 31: Distr...
4,5,53,0.2159,"Mr. Speaker, I now turn my attention to: the..."


13:33:06 | INFO | STAGE 2 ▶ Selecting top 2 of 5 chunks
13:33:06 | INFO | STAGE 2 ✓ Context length: 1005 chars
13:33:06 | INFO | STAGE 3 ▶ Building prompt
13:33:06 | INFO | STAGE 3 ✓ Prompt length: 1416 chars
13:33:06 | INFO | STAGE 4 ▶ Calling LLM...
13:33:06 | INFO | STAGE 4 ✓ Response from [smart-simulation] in 0.00s
13:33:06 | INFO | STAGE 1 ▶ Retrieval started
13:33:06 | INFO | STAGE 1 ✓ Retrieved 5 docs in 0.004s



✂️  STAGE 2 — CONTEXT SELECTION (keeping top 2)
-----------------------------------------------------------------
  [Doc 59] Score=0.3465
  ese economies continue to struggle with structural weaknesses in the business environment and governance. Mr. Speaker, growth in S SA is projected to strengthen to 4.1 percent in 2025 and 4.3 percent in 2026, supported by easing financial conditions and continued declines i n inflation. In sum, nearly half of SSA economies are expected to experience improved growth trajectories over the next two years. Resetting the Economy for the Ghana We Want 2025 Budget 6 Table 1: Global Economic Growth Rate

  [Doc 70] Score=0.3225
  1,800 1,801 1,850 2,388 2,325 3 Cocoa Beans (Mt) 2,341 2,370 2,430 2,390 3,200 7,330 6,000 Source: World Bank Commodity Markets Outlook, October, 2024. * Projected outturn, **Projection Resetting the Economy for the Ghana We Want 2025 Budget 8 ECOWAS Sub-Region: Macroeconomic Developments 54. Mr. Speaker, the Economic Community o

...

13:33:06 | INFO | STAGE 2 ▶ Selecting top 2 of 5 chunks
13:33:06 | INFO | STAGE 2 ✓ Context length: 1005 chars
13:33:06 | INFO | STAGE 3 ▶ Building prompt
13:33:06 | INFO | STAGE 3 ✓ Prompt length: 1413 chars
13:33:06 | INFO | STAGE 4 ▶ Calling LLM...
13:33:06 | INFO | STAGE 4 ✓ Response from [smart-simulation] in 0.02s
13:33:06 | INFO | STAGE 1 ▶ Retrieval started
13:33:06 | INFO | STAGE 1 ✓ Retrieved 5 docs in 0.002s



✂️  STAGE 2 — CONTEXT SELECTION (keeping top 2)
-----------------------------------------------------------------
  [Doc 4] Score=0.3799
  2025 AND MEDIUM-TERM POLICY OBJECTIVES AND TARGETS   48 Introduction         48 Fiscal Policy Objectives       49 2025 Fiscal Framework       50 2025 Macroeconomic Targets       51 2025 Expenditure Measures       51 2025 Revenue Measures       53 2025 Resource Allocation       58 2025 Budget Balances and Financing Operations     59 2025 Petroleum Receipts and Utilisation Projection     60 2025 Debt Policy Objectives and Liability Management     61 2025 Key Initiatives and Outlook for the Ghana S

  [Doc 53] Score=0.1823
  Mr. Speaker, I now turn my attention to:   the state of the Ghanaian economy in 2024;   macroeconomic policies, targets, and measures for 2025 and the medium-term;   policy initiatives and budget allocations; and   sectoral performance and outlook. Resetting the Economy for the Ghana We Want 2025 Budget 5 SECTION 2: GLOBAL ECONOMI

,Rank,Doc ID,Similarity Score,Preview
0,1,24,0.2771,Direct Investment FoBs Forward Operating Base...
1,2,0,0.1897,THEME: Resetting The Economy For The Ghana We ...
2,3,25,0.1723,rary Authority GNCCI Ghana National Chamber of...
3,4,77,0.1598,an improvement compared to the preceding year....
4,5,1,0.1303,AND ECONOMIC POLICY OF GOVERNMENT The Budget S...


13:33:06 | INFO | STAGE 2 ▶ Selecting top 2 of 5 chunks
13:33:06 | INFO | STAGE 2 ✓ Context length: 1005 chars
13:33:06 | INFO | STAGE 3 ▶ Building prompt
13:33:06 | INFO | STAGE 3 ✓ Prompt length: 1407 chars
13:33:06 | INFO | STAGE 4 ▶ Calling LLM...
13:33:06 | INFO | STAGE 4 ✓ Response from [smart-simulation] in 0.00s



✂️  STAGE 2 — CONTEXT SELECTION (keeping top 2)
-----------------------------------------------------------------
  [Doc 24] Score=0.2771
   Direct Investment FoBs Forward Operating Bases Resetting the Economy for the Ghana We Want 2025 Budget vii GAEC Ghana Atomic Energy Commission GHANEPS Ghana Electronic Procurement System GFSF Ghana Financial Stability Fund GFIM Ghana Fixed Income Market GOLDBOD Ghana Gold Board GIS Ghana Immigration Service GIIF Ghana Infrastructure Investment Fund GIPC Ghana Investment Promotion Centre GLMIS Ghana Labour Market Information System GhLA Ghana Library Authority GNCCI Ghana National Chamber of Com

  [Doc 0] Score=0.1897
  THEME: Resetting The Economy For The Ghana We Want The Budget Statement and Economic Policy of the Government of Ghana for the 2025 Financial Year Presented to Parliament by DR. CASSIEL ATO FORSON (MP) MINISTER FOR FINANCE On Tuesday March 11, 2025 On the Authority of HIS EXCELLENCY JOHN DRAMANI MAHAMA PRESIDENT OF THE REPUBLIC OF